# Collect the C-SPAN Abstracts as the Summarization Reference

| Item | Description |
|---|---|
| Goal | Collect C-SPAN's editorial abstract for every congressional committee program in its Video Library. The abstracts are written by people, about the same hearings, with no knowledge of this project, so they are the independent reference the generated summaries are checked against. The median abstract is 48 words, so it supports a nearest-abstract test, not a word-overlap score. |
| Input | The C-SPAN Archives API, `/2.0/programs/search` and `/2.0/programs/{id}`. The key `CSPAN_API_KEY` in `.env` in the D folder; it is requested from api@c-spanarchives.org and is not self-serve. |
| Output | In the D2 scratch folder: `Raw/search/<category>_<year>.jsonl`, one line per program found; `Raw/detail/<videoId>.json`, one file per program, only when `FETCH_DETAIL` is on; `Processed/programs.parquet`, one row per program with `id`, `title`, `description`, `date`, `categories`, and `word_count`. |
| Run | Last run 2026-08-20; the cell outputs shown are from that run. Not rerun: the reference set is complete and T1 was evaluated against it. The files now sit in the folders named above. |

## Step 1. Import packages

In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

## Step 2. Define file locations

One file per category and year under `Raw/search`, so a slice already on disk is never fetched again.

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D - Data Collection and Processing/D2 - C-SPAN Reference Abstracts")
NOTEBOOK_NAME = "D2.1 - C-SPAN - Collect Reference Abstracts"
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
SEARCH_FOLDER = RAW_FOLDER / "search"  # one JSONL per category and year
DETAIL_FOLDER = RAW_FOLDER / "detail"  # one JSON per program, only when FETCH_DETAIL is on
ENV_FILE = PROJECT_FOLDER / "D - Data Collection and Processing" / ".env"  # holds CSPAN_API_KEY

for folder in (SEARCH_FOLDER, DETAIL_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)

## Step 3. Set the collection window and read the API key

C-SPAN began broadcasting in 1979. The per-program detail call adds `publicId`, `eventid`, `videoDuration`, and `time`, at one request per program. The two fields that justified it did not survive checking. The search's `id` already is the video id, and `eventid` marks a broadcast event, not a hearing, so the detail call is off by default.

In [1]:
FIRST_YEAR = 1979
LAST_YEAR = datetime.now(timezone.utc).year
CATEGORIES = ("House Committee", "Senate Committee", "Joint Committee")  # the three values that hold hearings; there is no combined one
FETCH_DETAIL = False

API_KEY = os.environ.get("CSPAN_API_KEY")
if not API_KEY and ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():  # one KEY=value per line
        name, _, value = line.strip().partition("=")
        if name == "CSPAN_API_KEY" and value:
            API_KEY = value.strip().strip('"').strip("'")
assert API_KEY, f"CSPAN_API_KEY is not set and {ENV_FILE} does not supply it; request a key from api@c-spanarchives.org"

print(f"categories : {CATEGORIES}")
print(f"years      : {FIRST_YEAR}-{LAST_YEAR}")
print(f"search  -> {SEARCH_FOLDER}")
print(f"detail  -> {DETAIL_FOLDER}")

categories : ('House Committee', 'Senate Committee', 'Joint Committee')
years      : 1979-2026
search  -> /scratch2/acroh/projects/llm_method/data/raw/cspan/search
detail  -> /scratch2/acroh/projects/llm_method/data/raw/cspan/detail


## Step 4. Define one API request and one search walk

A request carries the key in a header, waits half a second between calls, and retries a transient failure up to five times. The search returns 20 programs per page and no total count, so the walk cannot check itself the way D1.1 does. Instead it refuses to loop: a cursor that returns only programs already seen, or stops advancing, stops the notebook. Dates must be full timestamps, because the shorter forms return a server error.

In [ ]:
BASE_URL = "https://api.c-spanarchives.org/2.0"
SECONDS_BETWEEN_REQUESTS = 0.5  # no published limit; two per second is slow enough not to matter to C-SPAN
MAX_ATTEMPTS = 5
TRANSIENT_STATUS = {408, 425, 429, 500, 502, 503, 504}

session = requests.Session()
request_log = {"requests": 0, "retries": 0, "last_request": 0.0}


def cspan_get(path, params=None):
    """One GET, returned as parsed JSON. Retries a transient failure, raises on anything else."""
    url = f"{BASE_URL}{path}"
    last_error = "no attempt made"
    for attempt in range(MAX_ATTEMPTS):
        wait = SECONDS_BETWEEN_REQUESTS - (time.monotonic() - request_log["last_request"])
        if wait > 0:
            time.sleep(wait)
        request_log["last_request"] = time.monotonic()
        request_log["requests"] += 1
        try:
            response = session.get(url, params=params or {}, headers={"x-api-key": API_KEY, "Accept": "application/json"}, timeout=60)
        except requests.RequestException as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        else:
            if response.status_code == 200:
                return response.json()
            if response.status_code not in TRANSIENT_STATUS:
                raise RuntimeError(f"HTTP {response.status_code} for {url}: {response.text[:200]}")
            last_error = f"HTTP {response.status_code}"
        request_log["retries"] += 1
        time.sleep(2.0 ** attempt)
    raise RuntimeError(f"{MAX_ATTEMPTS} attempts failed for {url}: {last_error}")


def search_year(year, category):
    """Every program in one category and one calendar year, following the cursor to the end."""
    query = f'category:"{category}" AND date:[{year}-01-01T00:00:00Z TO {year}-12-31T23:59:59Z]'
    programs = []
    seen = set()
    cursor = None
    while True:
        params = {"query": query, "sort": "date asc"}
        if cursor:
            params["cursor"] = cursor
        payload = cspan_get("/programs/search", params)
        page = payload.get("programs") or []
        if not page:
            break
        fresh = [p for p in page if p.get("id") not in seen]
        assert fresh, f"the cursor returned only programs already seen for {query!r}; the walk is looping"
        for program in fresh:
            seen.add(program["id"])
            programs.append(program)
        next_cursor = payload.get("cursor")
        if not next_cursor or next_cursor == cursor:
            break
        cursor = next_cursor
    return programs


def program_record(video_id):
    """One program's full record: adds publicId, eventid, videoDuration, and time to what the search returns."""
    return cspan_get(f"/programs/{video_id}")

## Step 5. Search every category and year

One cursor walk per category and year, written to its own file. A slice already on disk is read back instead of fetched, so the notebook is safe to interrupt and run again. An empty year is written as an empty file, so a later run can tell "collected, nothing there" from "never collected".

In [2]:
def slice_path(category, year):
    return SEARCH_FOLDER / f"{category.replace(' ', '_').lower()}_{year}.jsonl"


rows = []
for category in CATEGORIES:
    per_year = []
    for year in range(FIRST_YEAR, LAST_YEAR + 1):
        path = slice_path(category, year)
        if path.exists():
            programs = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
        else:
            programs = search_year(year, category)
            path.write_text("".join(json.dumps(p) + "\n" for p in programs))
        for p in programs:
            rows.append({**p, "category": category, "query_year": year})
        per_year.append((year, len(programs)))

    total = sum(n for _, n in per_year)
    first = next((y for y, n in per_year if n), None)
    print(f"{category:18} {total:>6,} programs, first year with any: {first}")

search = pd.DataFrame(rows)
print(f"\n{len(search):,} rows, {search['id'].nunique():,} distinct programs")
print(f"{request_log['requests']:,} API requests, {request_log['retries']} retries")

House Committee     8,396 programs, first year with any: 1980
Senate Committee    9,954 programs, first year with any: 1981
Joint Committee       347 programs, first year with any: 1983



18,697 rows, 18,697 distinct programs
0 API requests, 0 retries


## Step 6. Count programs by year and category

A program can sit in more than one category, so the distinct count is the one that matters. A year that drops to a fraction of its neighbours is a truncated walk before it is a quiet year in Congress.

In [3]:
by_year = search.pivot_table(index="query_year", columns="category", values="id", aggfunc="nunique", fill_value=0)
by_year["distinct"] = search.groupby("query_year")["id"].nunique()
print(by_year.to_string())

category    House Committee  Joint Committee  Senate Committee  distinct
query_year                                                              
1980                      1                0                 0         1
1981                      0                0                19        19
1982                      0                0                 7         7
1983                     14                1                19        34
1984                      5                0                13        18
1985                     40                2                47        89
1986                     59                8                66       133
1987                     69               56               127       252
1988                    128               11               140       279
1989                    198                7               189       394
1990                    192               14               222       428
1991                    192               12       

## Step 7. Fetch the full record for each program, when enabled

Skipped unless `FETCH_DETAIL` is on. One request per program, so this is the long part of the notebook when it runs.

In [4]:
ids = sorted(search["id"].dropna().astype(int).unique())
missing = [i for i in ids if not (DETAIL_FOLDER / f"{i}.json").exists()] if FETCH_DETAIL else []
if not FETCH_DETAIL:
    print(f"FETCH_DETAIL is off; {len(list(DETAIL_FOLDER.glob('*.json'))):,} detail records already on disk are used where present")
else:
    print(f"{len(ids):,} distinct programs, {len(missing):,} still to fetch (~{len(missing) * SECONDS_BETWEEN_REQUESTS / 60:.0f} min)")

failures = []
for n, video_id in enumerate(missing, 1):
    try:
        record = program_record(video_id)
    except RuntimeError as exc:
        failures.append({"id": video_id, "error": str(exc)[:200]})
        continue
    (DETAIL_FOLDER / f"{video_id}.json").write_text(json.dumps(record))
    if n % 2000 == 0:
        print(f"  {n:,}/{len(missing):,}")

print(f"done. {len(failures)} failed")
if failures:
    print(pd.DataFrame(failures).head(10).to_string(index=False))

FETCH_DETAIL is off; 1,331 detail records already on disk are used where present
done. 0 failed


## Step 8. Merge and save

The search is the spine: every program has a search row, and only some have a detail record. Where both carry a field the detail record wins.

In [5]:
detail_files = sorted(DETAIL_FOLDER.glob("*.json"))
if detail_files:
    detail = pd.DataFrame(json.loads(p.read_text()) for p in detail_files)
else:
    detail = pd.DataFrame(columns=["id"])
print(f"detail records available: {len(detail):,}")

categories = search.groupby("id")["category"].unique().map(sorted).str.join("|").rename("categories")

base = search.drop_duplicates("id").drop(columns=["category", "query_year"])
extra = [c for c in detail.columns if c not in base.columns and c != "id"]
programs = base
if extra:
    programs = programs.merge(detail[["id", *extra]], on="id", how="left")
programs = programs.merge(categories, left_on="id", right_index=True, how="left")
programs["word_count"] = programs["description"].fillna("").str.split().str.len()
programs = programs.sort_values("date").reset_index(drop=True)
programs.to_parquet(PROCESSED_FOLDER / "programs.parquet", index=False)

print(f"\nwrote {PROCESSED_FOLDER / 'programs.parquet'}  ({len(programs):,} programs)")
print(f"date span: {programs['date'].min()[:10]} to {programs['date'].max()[:10]}")
have = programs["description"].fillna("").str.strip().ne("")
print(f"with an abstract: {have.sum():,} ({have.mean():.1%})")
print(f"abstract words: median {int(programs.loc[have, 'word_count'].median())}, p90 {int(programs.loc[have, 'word_count'].quantile(0.9))}")

detail records available: 1,331



wrote /scratch2/acroh/projects/llm_method/data/raw/cspan/programs.parquet  (18,697 programs)
date span: 1980-10-01 to 2026-08-06
with an abstract: 17,838 (95.4%)
abstract words: median 51, p90 101


## Step 9. Check the table

Four checks. The first three stop the notebook if they fail.

- One row per program.
- Every searched program is in the table.
- Every program's `date` falls inside the year it was queried under.
- Programs without an abstract are counted by year, not asserted away, since they cannot serve as a reference.

In [6]:
dupes = int(programs["id"].duplicated().sum())
assert dupes == 0, f"{dupes} duplicate program ids after the merge"
print(f"1) {len(programs):,} rows, one per program")

searched = set(search["id"].dropna().astype(int))
fetched = set(programs["id"].dropna().astype(int))
gap = searched - fetched
assert not gap, f"{len(gap):,} searched programs are missing from the table, e.g. {sorted(gap)[:5]}"
print(f"2) all {len(searched):,} searched programs are in the table")

year_of_slice = search.drop_duplicates("id").set_index("id")["query_year"]
issued = pd.to_datetime(programs["date"], format="mixed", utc=True).dt.year
off = int((issued.values != programs["id"].map(year_of_slice).values).sum())
assert off == 0, f"{off} programs fall outside the year they were queried under"
print("3) every program's date falls inside its query year")

no_abstract = programs.loc[~have]
print(f"4) {len(no_abstract):,} programs have no abstract ({len(no_abstract) / len(programs):.1%})")
if len(no_abstract):
    print(no_abstract["date"].str[:4].rename("y").value_counts().sort_index().rename("n").to_string())

1) 18,697 rows, one per program
2) all 18,697 searched programs are in the table
3) every program's date falls inside its query year
4) 859 programs have no abstract (4.6%)
y
1981      2
1982      1
1983      1
1985      1
1986      1
1987      3
1988      2
1989      8
1990      3
1991      1
2001      2
2002      4
2003     14
2004      5
2005      2
2012      2
2013      9
2014    127
2015    483
2016     95
2017      8
2018      6
2019      1
2020      3
2021      5
2022      5
2023     38
2024     11
2025      4
2026     12
